# Width distillation: 420 pd_N8 → 256 / 128 at the fixed 8-step schedule

Prerequisite: `checkpoints_fm/best_420_pd_N8.pt` from the progressive-distillation cell
(v-parametrized, gated visually at 8 NFE).

Idea: the light net does **not** learn the continuous field (that plateaued at every width).
It only has to match the 420 teacher's `v` at the **8 grid times** the sampler will query.

- Teacher: 420 non-EMA `student` from `pd_N8`, frozen (EMA lagged at PD step counts).
- Student: cold 256 or 128 (set `WIDTH`).
- States: `z_t = α_t x1 + σ_t ε`, fresh ε, `t ∈ {1/8, …, 1}` — plus a fraction of
  **rollout states** (teacher v-DDIM partway from noise), because an 8-step sampler
  compounds errors on exactly those off-interpolant states.
- Loss: same SNR-weighted x-space v-matching as the PD cell. No divisions in the student path.

Outputs: `checkpoints_fm/best_{WIDTH}_w8.pt` (`param='v'`, `nfe=8`) — loadable by `check_sampling.ipynb`.

Run this notebook once with `WIDTH=256`, once with `WIDTH=128`. Judge each by the sampling
cell below and CPU latency, not by the loss scalar.

In [ ]:
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
WIDTH = 256              # then rerun with 128
BATCH = 256
EPOCHS = 20
LR = 1e-4
EMA_DECAY = 0.999
NFE = 8
TEACHER_T = 100          # γ table resolution (same as all prior runs)
NOISE_PRECISION = 1e-5
ROLLOUT_P = 0.3          # fraction of batches drawn from teacher rollout states
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"          # context norms only
TEACHER_CKPT = Path("./checkpoints_fm/best_420_pd_N8.pt")
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{WIDTH}_w8.log"
DT = 1.0 / NFE


# --------------------- boilerplate ---------------------
class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
    ) * node_mask


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


# --------------------- schedule + v-param ---------------------
def alpha_sigma(t, sched, like):
    T = sched.timesteps
    u = (t * T).clamp(0, T)
    i0 = u.floor().long().clamp(0, T - 1)
    frac = (u - i0.float()).clamp(0.0, 1.0)
    g = sched.gamma
    gamma = g[i0] + frac * (g[i0 + 1] - g[i0])
    a = torch.sqrt(torch.sigmoid(-gamma)).view(like.shape[0], 1, 1)
    s = torch.sqrt(torch.sigmoid(gamma)).view(like.shape[0], 1, 1)
    return a, s


def x_eps_from_v(z, v, a, s):
    return a * z - s * v, s * z + a * v


def ddim_v(z, v, a_t, s_t, a_s, s_s, nm):
    x, eps = x_eps_from_v(z, v, a_t, s_t)
    return com_project(a_s * x + s_s * eps, nm)


def snr_weight(a, s):
    return torch.clamp(a * a / (s * s).clamp_min(1e-8), min=1.0)


def x_loss(x_pred, x_tgt, w, nm):
    err = (x_pred - x_tgt) ** 2 * nm * w
    n = nm.sum().clamp_min(1)
    lx = err[..., :3].sum() / (n * 3)
    lh = err[..., 3:].sum() / (n * 8)
    return lx + lh, lx.detach(), lh.detach()


# --------------------- models ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION).to(device)

tc = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
assert tc.get("param") == "v" and int(tc.get("nfe", 0)) == NFE, (
    f"{TEACHER_CKPT.name} must be the v-param pd_N{NFE} checkpoint"
)
teacher = EquivariantFlowMatching(
    EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=int(tc["hidden_nf"]), device=device),
    in_node_nf=8,
).to(device)
teacher.load_state_dict(tc["student"])  # non-EMA: visually better, EMA lagged at PD step counts
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = EquivariantFlowMatching(
    EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=WIDTH, device=device),
    in_node_nf=8,
).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(3000.0)

GRID = torch.linspace(1.0, DT, NFE, device=device)   # t = 1, 7/8, ..., 1/8


@torch.no_grad()
def rollout_states(B, N, nm, em, bctx):
    """Teacher v-DDIM from noise for k ∈ [1, NFE-1] steps → (z_t, t) the sampler will visit."""
    z = com_project(torch.randn(B, N, 11, device=device) * nm, nm)
    k = int(torch.randint(1, NFE, (1,)).item())
    ts = torch.linspace(1.0, 0.0, NFE + 1, device=device)
    for i in range(k):
        t0 = ts[i].expand(B, 1)
        t1 = ts[i + 1].expand(B, 1)
        a_t, s_t = alpha_sigma(t0, sched, z)
        a_s, s_s = alpha_sigma(t1, sched, z)
        v = teacher.velocity(z, t0, nm, em, bctx)
        z = ddim_v(z, v, a_t, s_t, a_s, s_s, nm)
    t = ts[k].expand(B, 1)
    return z, t


log(f"start width-distill {int(tc['hidden_nf'])}→{WIDTH} nfe={NFE} rollout_p={ROLLOUT_P} batch={BATCH}")
best, no_imp = float("inf"), 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    x1_all, na_all, ctx_all = load_part_pairs(part_i)
    perm = torch.randperm(x1_all.shape[0])
    x1_all, na_all, ctx_all = x1_all[perm], na_all[perm], ctx_all[perm]
    student.train()
    run, rx, rh, ns = 0.0, 0.0, 0.0, 0
    pbar = tqdm(range(0, x1_all.shape[0] - BATCH + 1, BATCH), desc=f"w{WIDTH} ep{epoch}")
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = na_all[sl].to(device=device, dtype=torch.long)
        nm, em = prepare_masks(na, PAD_TO, device)
        ctx = ctx_all[sl].to(device=device, dtype=torch.float32)
        bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
        x1_b = com_project(x1_all[sl].to(device=device, dtype=torch.float32), nm)
        B, N = x1_b.shape[:2]

        if torch.rand(()) < ROLLOUT_P:
            zt, t = rollout_states(B, N, nm, em, bctx)
        else:
            t = GRID[torch.randint(0, NFE, (B,), device=device)].unsqueeze(1)
            eps = student.sample_combined_position_feature_noise(B, N, nm)
            a_t, s_t = alpha_sigma(t, sched, x1_b)
            zt = com_project(a_t * x1_b + s_t * eps, nm)

        a_t, s_t = alpha_sigma(t, sched, zt)
        with torch.no_grad():
            v_tea = teacher.velocity(zt, t, nm, em, bctx)
            x_tgt, _ = x_eps_from_v(zt, v_tea, a_t, s_t)
        v_pred = student.velocity(zt, t, nm, em, bctx)
        x_pred, _ = x_eps_from_v(zt, v_pred, a_t, s_t)
        loss, lx, lh = x_loss(x_pred, x_tgt, snr_weight(a_t, s_t), nm)
        if not torch.isfinite(loss):
            log(f"WARN non-finite ep={epoch}")
            continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, q)
        opt.step()
        ema.update_model_average(student_ema, student)
        run += loss.item(); rx += lx.item(); rh += lh.item(); ns += 1
        if ns % 20 == 0:
            pbar.set_postfix(loss=f"{run/ns:.5f}", lx=f"{rx/ns:.5f}", lh=f"{rh/ns:.5f}")

    avg = run / max(ns, 1)
    log(f"epoch={epoch} avg={avg:.6f} lx={rx/max(ns,1):.6f} lh={rh/max(ns,1):.6f}")
    ckpt = {
        "epoch": epoch, "avg_loss": avg, "stage": f"w{WIDTH}",
        "nfe": NFE, "param": "v", "hidden_nf": WIDTH,
        "teacher": TEACHER_CKPT.name,
        "student": student.state_dict(), "student_ema": student_ema.state_dict(),
        "opt": opt.state_dict(),
    }
    torch.save(ckpt, CKPT_DIR / f"latest_{WIDTH}_w8.pt")
    if avg < best - MIN_DELTA:
        best, no_imp = avg, 0
        torch.save(ckpt, CKPT_DIR / f"best_{WIDTH}_w8.pt")
        log(f"ckpt best avg={avg:.6f}")
    else:
        no_imp += 1
        best = min(best, avg)
    if no_imp >= EARLY_STOP_PATIENCE:
        log(f"early stop epoch={epoch}")
        break

log(f"done width={WIDTH} best={best:.6f}")

In [ ]:
"""Quick gate: teacher(420 pd_N8) vs this student at 8 NFE, same contexts."""
import math
import time

import py3Dmol
from rdkit import Chem

try:
    from src.mlconfgen.utils import ATOM_DECODER, align_mol_to_principal_frame, prepare_edm_input, samples_to_rdkit_mol
except ImportError:
    from ml_conformer_generator.src.mlconfgen.utils import (
        ATOM_DECODER, align_mol_to_principal_frame, prepare_edm_input, samples_to_rdkit_mol,
    )

MOL_PATH = "./assets/demo_files/ceyyag.mol"
N_SHOW = 4
torch.manual_seed(42)


def show(ms):
    blocks = [Chem.MolToXYZBlock(m) for m in ms if m is not None]
    cols = min(4, len(blocks))
    rows = math.ceil(len(blocks) / cols)
    v = py3Dmol.view(viewergrid=(rows, cols), width=250 * cols, height=250 * rows)
    for i, b in enumerate(blocks):
        r, c = divmod(i, cols)
        v.addModel(b, "xyz", viewer=(r, c))
        v.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}}, viewer=(r, c))
        v.zoomTo(viewer=(r, c))
    v.show()


@torch.inference_mode()
def sample8(model, nm, em, ctx, tag):
    B, N = nm.shape[:2]
    z = com_project(torch.randn(B, N, 11, device=nm.device) * nm, nm)
    ts = torch.linspace(1.0, 0.0, NFE + 1, device=nm.device)
    t0_wall = time.perf_counter()
    for i in range(NFE):
        t = ts[i].expand(B, 1)
        s = ts[i + 1].expand(B, 1)
        a_t, s_t = alpha_sigma(t, sched, z)
        a_s, s_s = alpha_sigma(s, sched, z)
        v = model.velocity(z, t, nm, em, ctx)
        z = ddim_v(z, v, a_t, s_t, a_s, s_s, nm)
    dt_wall = time.perf_counter() - t0_wall
    x = remove_mean_with_mask(z[..., :3], nm)
    h = torch.nn.functional.one_hot(z[..., 3:].argmax(-1), 8).float() * nm
    print(f"{tag}: {dt_wall*1000/B:.1f} ms/mol  coord std={x[nm[...,0].bool()].std().item():.3f}")
    return x, h


ref = Chem.RemoveAllHs(Chem.MolFromMolFile(MOL_PATH))
ctx3, *_ = align_mol_to_principal_frame(ref)
na_ref = ref.GetNumAtoms()
nm_s, em_s, ctx_s = prepare_edm_input(
    N_SHOW, ctx3.to(device), norms, na_ref, na_ref, device, pad_to=PAD_TO
)

print("teacher (420 pd_N8)")
xt, ht = sample8(teacher, nm_s, em_s, ctx_s, "teacher")
show(samples_to_rdkit_mol(xt.cpu(), ht.cpu(), nm_s.cpu(), ATOM_DECODER))

# non-EMA first (was better in the 420 PD run); swap in student_ema to compare
print(f"student ({WIDTH}, non-EMA)")
student.eval()
xs, hs = sample8(student, nm_s, em_s, ctx_s, f"student{WIDTH}")
show(samples_to_rdkit_mol(xs.cpu(), hs.cpu(), nm_s.cpu(), ATOM_DECODER))

# Warm-started width distillation (420 → 256 / 128)

Both cold-start width runs (8-step and 32-step grids) stalled at the same loss ≈ 0.10 —
the wall is the cold start, not hop size. The only student that ever converged here (420)
was warm-started. This cell warm-starts the small student by **slicing the 420 teacher's
weights**, then fine-tunes with the same width-distill objective.

How the init works:

- The node feature vector `h` is a **residual stream** shared by all 9 blocks
  (`node_mlp` output adds to it), so one global set of `WIDTH` residual channels is chosen
  by aggregate weight magnitude and sliced consistently in every layer that reads or
  writes `h` (embedding, edge/node/coord MLP inputs, node MLP output, output head).
- Per-MLP internal spaces (edge hidden, message, node hidden, coord hidden) are
  independent, so their neurons are ranked and sliced per layer (in-norm + out-norm).
- No rescaling is applied for the removed channels — the first epochs of fine-tuning fix
  the resulting activation shrinkage.
- **The two output heads are shrunk ×0.05 after slicing** (per-block `coord_mlp[4]` and
  `embedding_out`). This is load-bearing: a raw slice looked fine on a random-noise probe
  but exploded on molecule-like interpolant states (first-batch loss ≈ 5900, worst-case
  x-loss ≈ 1e3 — coordinate updates compound across the 9 blocks). Cold init is stable
  for exactly this reason (its coord head starts at gain 1e-3). With ×0.05 heads the
  start loss matches cold (≈ 0.64 vs 0.67 measured) but the features underneath are the
  teacher's, which is where the warm-start value lives.

Defaults target the actual goal directly: `NFE=8` against the `pd_N8` teacher — the
parent model already solves this exact task, and fine-tuning only has to repair the
truncation damage. The first-batch loss is logged before any update: with the ×0.05
heads it should start ≈ 0.5–0.7 (same as cold — anything ≫ 1 means the init is broken
again). The win to watch for is the *descent rate*: if it does not drop clearly below
the cold-start 0.10 plateau by epoch 3–4, fall back to `NFE=32` + the `pd_N32` teacher
and halve later.

**Measured on the real `best_420_pd_N8` checkpoint** (random-noise velocity probe,
cosine to teacher / relative MSE — note this probe under-detects the head problem above,
which only shows on molecule-like states):

| init | cosine | rel. MSE |
|---|---|---|
| 420→256 sliced | **0.45** | 0.86 |
| 256 cold | 0.02 | 1.00 |
| 420→128 sliced | 0.06 | **12.9 (broken)** |
| 128 cold | −0.02 | 1.00 |

So: run `WIDTH=256` with the default source first. **Do not slice 420→128 directly** —
keeping only 30% of channels wrecks the output scale. For 128, set
`SOURCE_CKPT = "./checkpoints_fm/best_256_ws8.pt"` (the fine-tuned 256; a 50% keep) while
leaving the 420 `pd_N8` as the target teacher. Per-row norm re-scaling of sliced weights
was tested and rejected — amplification compounds across the 9 blocks and diverges.

Writes `best_{WIDTH}_ws{NFE}.pt` (same schema; loadable by `check_sampling.ipynb` and the
gate cell above).

In [ ]:
"""Warm-started width distillation: slice 420 -> WIDTH, then fine-tune on the grid."""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
WIDTH = 256              # then 128
NFE = 8                  # 8 + pd_N8 teacher; fallback: 32 + pd_N32
# Warm-start source. None -> slice the teacher (validated for 420->256).
# For WIDTH=128 point this at the FINE-TUNED 256 (direct 420->128 slicing is broken):
#   SOURCE_CKPT = "./checkpoints_fm/best_256_ws8.pt"
SOURCE_CKPT = None
BATCH = 64
EPOCHS = 16
LR = 1e-4
EMA_DECAY = 0.99         # short window (PD lesson: 0.999 never catches up)
ROLLOUT_P = 0.3
TEACHER_T = 100
NOISE_PRECISION = 1e-5
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
_cands = [Path(f"./checkpoints_fm/420_pd_vd/best_420_pd_N{NFE}.pt"),
          Path(f"./checkpoints_fm/best_420_pd_N{NFE}.pt")]
TEACHER_CKPT = next((p for p in _cands if p.exists()), _cands[0])
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{WIDTH}_ws{NFE}.log"
DT = 1.0 / NFE


# --------------------- boilerplate ---------------------
class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
    ) * node_mask


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def alpha_sigma(t, sched, like):
    T = sched.timesteps
    u = (t * T).clamp(0, T)
    i0 = u.floor().long().clamp(0, T - 1)
    frac = (u - i0.float()).clamp(0.0, 1.0)
    g = sched.gamma
    gamma = g[i0] + frac * (g[i0 + 1] - g[i0])
    a = torch.sqrt(torch.sigmoid(-gamma)).view(like.shape[0], 1, 1)
    s = torch.sqrt(torch.sigmoid(gamma)).view(like.shape[0], 1, 1)
    return a, s


def x_eps_from_v(z, v, a, s):
    return a * z - s * v, s * z + a * v


def ddim_v(z, v, a_t, s_t, a_s, s_s, nm):
    x, eps = x_eps_from_v(z, v, a_t, s_t)
    return com_project(a_s * x + s_s * eps, nm)


def snr_weight(a, s):
    return torch.clamp(a * a / (s * s).clamp_min(1e-8), min=1.0)


def x_loss(x_pred, x_tgt, w, nm):
    err = (x_pred - x_tgt) ** 2 * nm * w
    n = nm.sum().clamp_min(1)
    lx = err[..., :3].sum() / (n * 3)
    lh = err[..., 3:].sum() / (n * 8)
    return lx + lh, lx.detach(), lh.detach()


# --------------------- warm start: slice 420 -> WIDTH ---------------------
def _row_norm(W):
    return W.pow(2).sum(1)


def _col_norm(W):
    return W.pow(2).sum(0)


def _topk(score, k):
    return torch.topk(score, k).indices.sort().values


@torch.no_grad()
def warm_start_from(big_dyn, small_dyn, Hb, Hs, head_scale=0.05):
    """Init small EGNNDynamics by magnitude-ranked channel slicing of the big one.

    The residual stream (node features h) is sliced with ONE global index set S,
    used consistently in every layer that touches h. Internal MLP spaces are
    ranked and sliced per layer.

    The two output heads (per-block coord_mlp[4] and embedding_out) are then
    shrunk by `head_scale`: sliced features are damaged, and full-magnitude
    heads driving them explode on molecule-like states (measured worst-case
    x-loss ~1e3 vs ~0.24 cold). At 0.05 the start loss matches a cold init
    while every block keeps the teacher's features.
    """
    bg, sm = big_dyn.egnn, small_dyn.egnn
    blocks_b = [getattr(bg, f"e_block_{i}") for i in range(9)]
    blocks_s = [getattr(sm, f"e_block_{i}") for i in range(9)]
    dev = bg.embedding.weight.device

    # global residual channels: aggregate importance over all readers/writers of h
    score = _row_norm(bg.embedding.weight) + _col_norm(bg.embedding_out.weight)
    for blk in blocks_b:
        for g in (blk.gcl_0, blk.gcl_1):
            W0 = g.edge_mlp[0].weight
            score += _col_norm(W0[:, :Hb]) + _col_norm(W0[:, Hb:2 * Hb])
            score += _col_norm(g.node_mlp[0].weight[:, :Hb])
            score += _row_norm(g.node_mlp[2].weight)
        Wc = blk.gcl_equiv.coord_mlp[0].weight
        score += _col_norm(Wc[:, :Hb]) + _col_norm(Wc[:, Hb:2 * Hb])
    S = _topk(score, Hs)

    def cp(dst, W, b=None):
        dst.weight.data.copy_(W)
        if b is not None and dst.bias is not None:
            dst.bias.data.copy_(b)

    # pair-input columns: [h_source(S), h_target(S), 2 edge feats]
    tail2 = torch.tensor([2 * Hb, 2 * Hb + 1], device=dev)
    pair_cols = torch.cat([S, S + Hb, tail2])

    cp(sm.embedding, bg.embedding.weight[S], bg.embedding.bias[S])
    cp(sm.embedding_out, bg.embedding_out.weight[:, S], bg.embedding_out.bias)

    for blk_b, blk_s in zip(blocks_b, blocks_s):
        for gb, gs in ((blk_b.gcl_0, blk_s.gcl_0), (blk_b.gcl_1, blk_s.gcl_1)):
            W0, b0 = gb.edge_mlp[0].weight, gb.edge_mlp[0].bias
            W2, b2 = gb.edge_mlp[2].weight, gb.edge_mlp[2].bias
            Wa, ba = gb.att_mlp[0].weight, gb.att_mlp[0].bias
            Wn0, bn0 = gb.node_mlp[0].weight, gb.node_mlp[0].bias
            Wn2, bn2 = gb.node_mlp[2].weight, gb.node_mlp[2].bias

            E = _topk(_row_norm(W0) + _col_norm(W2), Hs)              # edge hidden
            M = _topk(_row_norm(W2) + _col_norm(Wa)                   # message space
                      + _col_norm(Wn0[:, Hb:]), Hs)
            N = _topk(_row_norm(Wn0) + _col_norm(Wn2), Hs)            # node hidden

            cp(gs.edge_mlp[0], W0[E][:, pair_cols], b0[E])
            cp(gs.edge_mlp[2], W2[M][:, E], b2[M])
            cp(gs.att_mlp[0], Wa[:, M], ba)
            node_cols = torch.cat([S, M + Hb])                        # [h(S), agg(M)]
            cp(gs.node_mlp[0], Wn0[N][:, node_cols], bn0[N])
            cp(gs.node_mlp[2], Wn2[S][:, N], bn2[S])

        cb, cs = blk_b.gcl_equiv.coord_mlp, blk_s.gcl_equiv.coord_mlp
        W0, b0 = cb[0].weight, cb[0].bias
        W2, b2 = cb[2].weight, cb[2].bias
        W4 = cb[4].weight                                             # Linear(H,1,bias=False)
        C1 = _topk(_row_norm(W0) + _col_norm(W2), Hs)
        C2 = _topk(_row_norm(W2) + _col_norm(W4), Hs)
        cp(cs[0], W0[C1][:, pair_cols], b0[C1])
        cp(cs[2], W2[C2][:, C1], b2[C2])
        cs[4].weight.data.copy_(W4[:, C2])

    # shrink output heads toward the cold-init regime (see docstring)
    sm.embedding_out.weight.data.mul_(head_scale)
    sm.embedding_out.bias.data.mul_(head_scale)
    for blk in blocks_s:
        blk.gcl_equiv.coord_mlp[4].weight.data.mul_(head_scale)
    return S


# --------------------- models ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION).to(device)

tc = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
assert tc.get("param") == "v" and int(tc.get("nfe", 0)) == NFE, (
    f"{TEACHER_CKPT.name} must be the v-param pd_N{NFE} checkpoint"
)
HB = int(tc["hidden_nf"])
teacher = EquivariantFlowMatching(
    EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=HB, device=device),
    in_node_nf=8,
).to(device)
teacher.load_state_dict(tc["student"])   # non-EMA
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = EquivariantFlowMatching(
    EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=WIDTH, device=device),
    in_node_nf=8,
).to(device)

if SOURCE_CKPT is None or Path(SOURCE_CKPT) == TEACHER_CKPT:
    src_model, HS, src_name = teacher, HB, TEACHER_CKPT.name
else:
    sck = torch.load(SOURCE_CKPT, map_location=device, weights_only=False)
    HS, src_name = int(sck["hidden_nf"]), Path(SOURCE_CKPT).name
    src_model = EquivariantFlowMatching(
        EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=HS, device=device),
        in_node_nf=8,
    ).to(device)
    src_model.load_state_dict(sck["student"])
assert HS > WIDTH, f"source width {HS} must exceed student width {WIDTH}"
warm_start_from(src_model.dynamics, student.dynamics, HS, WIDTH)
log(f"warm-started {HS}->{WIDTH} from {src_name} (keep {WIDTH / HS:.0%} of channels)")
if src_model is not teacher:
    del src_model
    torch.cuda.empty_cache()

student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(100.0)  # tight initial clip: one bad early batch must not seed divergence

GRID = torch.linspace(1.0, DT, NFE, device=device)


@torch.no_grad()
def rollout_states(B, N, nm, em, bctx):
    z = com_project(torch.randn(B, N, 11, device=device) * nm, nm)
    k = int(torch.randint(1, NFE, (1,)).item())
    ts = torch.linspace(1.0, 0.0, NFE + 1, device=device)
    for i in range(k):
        t0 = ts[i].expand(B, 1)
        t1 = ts[i + 1].expand(B, 1)
        a_t, s_t = alpha_sigma(t0, sched, z)
        a_s, s_s = alpha_sigma(t1, sched, z)
        v = teacher.velocity(z, t0, nm, em, bctx)
        z = ddim_v(z, v, a_t, s_t, a_s, s_s, nm)
    return z, ts[k].expand(B, 1)


log(f"start WARM width-distill {HB}->{WIDTH} nfe={NFE} rollout_p={ROLLOUT_P} batch={BATCH}")
best, no_imp = float("inf"), 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    x1_all, na_all, ctx_all = load_part_pairs(part_i)
    perm = torch.randperm(x1_all.shape[0])
    x1_all, na_all, ctx_all = x1_all[perm], na_all[perm], ctx_all[perm]
    student.train()
    run, rx, rh, ns = 0.0, 0.0, 0.0, 0
    pbar = tqdm(range(0, x1_all.shape[0] - BATCH + 1, BATCH), desc=f"ws{WIDTH} ep{epoch}")
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = na_all[sl].to(device=device, dtype=torch.long)
        nm, em = prepare_masks(na, PAD_TO, device)
        ctx = ctx_all[sl].to(device=device, dtype=torch.float32)
        bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
        x1_b = com_project(x1_all[sl].to(device=device, dtype=torch.float32), nm)
        B, N = x1_b.shape[:2]

        if torch.rand(()) < ROLLOUT_P:
            zt, t = rollout_states(B, N, nm, em, bctx)
        else:
            t = GRID[torch.randint(0, NFE, (B,), device=device)].unsqueeze(1)
            eps = student.sample_combined_position_feature_noise(B, N, nm)
            a_t, s_t = alpha_sigma(t, sched, x1_b)
            zt = com_project(a_t * x1_b + s_t * eps, nm)

        a_t, s_t = alpha_sigma(t, sched, zt)
        with torch.no_grad():
            v_tea = teacher.velocity(zt, t, nm, em, bctx)
            x_tgt, _ = x_eps_from_v(zt, v_tea, a_t, s_t)
        v_pred = student.velocity(zt, t, nm, em, bctx)
        x_pred, _ = x_eps_from_v(zt, v_pred, a_t, s_t)
        loss, lx, lh = x_loss(x_pred, x_tgt, snr_weight(a_t, s_t), nm)
        if not torch.isfinite(loss):
            log(f"WARN non-finite ep={epoch}")
            continue
        if epoch == 0 and ns == 0:
            log(f"first-batch loss (pre-update, = warm-start quality): {loss.item():.4f} "
                f"(cold 256 started at ~0.5)")
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, q)
        opt.step()
        ema.update_model_average(student_ema, student)
        run += loss.item(); rx += lx.item(); rh += lh.item(); ns += 1
        if ns % 20 == 0:
            pbar.set_postfix(loss=f"{run/ns:.5f}", lx=f"{rx/ns:.5f}", lh=f"{rh/ns:.5f}")

    avg = run / max(ns, 1)
    log(f"epoch={epoch} avg={avg:.6f} lx={rx/max(ns,1):.6f} lh={rh/max(ns,1):.6f}")
    ckpt = {
        "epoch": epoch, "avg_loss": avg, "stage": f"ws{WIDTH}",
        "nfe": NFE, "param": "v", "hidden_nf": WIDTH,
        "teacher": TEACHER_CKPT.name, "warm_start": True,
        "student": student.state_dict(), "student_ema": student_ema.state_dict(),
        "opt": opt.state_dict(),
    }
    torch.save(ckpt, CKPT_DIR / f"latest_{WIDTH}_ws{NFE}.pt")
    if avg < best - MIN_DELTA:
        best, no_imp = avg, 0
        torch.save(ckpt, CKPT_DIR / f"best_{WIDTH}_ws{NFE}.pt")
        log(f"ckpt best avg={avg:.6f}")
    else:
        no_imp += 1
        best = min(best, avg)
    if no_imp >= EARLY_STOP_PATIENCE:
        log(f"early stop epoch={epoch}")
        break

log(f"done warm width={WIDTH} best={best:.6f}")